<a href="https://colab.research.google.com/github/Jgriffiths92/Open_E-Dope/blob/master/Kivy_App_To_APK.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Clean workspace from old remnants and legacy scripts
!rm -f /content/android-clang-wrapper.sh
!rm -rf ~/.buildozer/
!rm -rf .pytest_cache/
!rm -rf Open_E-Dope bin sample_data main.py buildozer.spec layout.kv extra_manifest.xml extra_intents.xml assets app res circularprogressbar.py

In [ ]:
# 2. Reset toxic compiler environment variables
import os
vars_to_clear = ['CC', 'CXX', 'CPATH', 'C_INCLUDE_PATH', 'CPLUS_INCLUDE_PATH', 'CFLAGS', 'CPPFLAGS', 'CXXFLAGS', 'PKG_CONFIG_PATH', 'PIP_CONSTRAINT']
for var in vars_to_clear:
    if var in os.environ:
        del os.environ[var]
print('Environment variables successfully reset.')

In [ ]:
# 3. Install standard system dependencies and modern toolchain
!sudo apt-get update
!sudo apt-get install -y build-essential libsqlite3-dev sqlite3 bzip2 libbz2-dev zlib1g-dev libssl-dev openssl libgdbm-dev libgdbm-compat-dev liblzma-dev libreadline-dev libncursesw5-dev libffi-dev uuid-dev
!apt-get install -y openjdk-17-jdk-headless
!pip install --upgrade pip
!pip install --upgrade pip Cython==0.29.37 buildozer virtualenv
!pip install --upgrade kivymd==1.2.0 pillow==11.3.0 plyer==2.1.0

In [ ]:
# 4. Fetch your project files from GitHub
!git clone https://github.com/Jgriffiths92/Open_E-Dope.git
!mv /content/Open_E-Dope/main.py  ../content/
!mv /content/Open_E-Dope/buildozer.spec  ../content/
!mv /content/Open_E-Dope/layout.kv  ../content/
!mv /content/Open_E-Dope/extra_manifest.xml  ../content/
!mv /content/Open_E-Dope/extra_intents.xml  ../content/
!mv /content/Open_E-Dope/assets  ../content/
!mv /content/Open_E-Dope/app  ../content/
!mv /content/Open_E-Dope/res  ../content/
!mv /content/Open_E-Dope/circularprogressbar.py  ../content/
print('Repository cloned and files moved to workspace root.')

### IMPORTANT STEP BEFORE COMPILING
Open `buildozer.spec` using Colab's left side panel and ensure your requirements line contains all libraries:
`requirements = hostpython3==3.11.9,python3==3.11.9,kivy==2.3.1,kivymd==1.2.0,pyjnius,pillow==11.3.0,plyer==2.1.0`

In [ ]:
# 5. Install API 33 into the exact SDK directory used by Buildozer
import os
for var in ('P4A_pyjnius_DIR', 'P4A_PYJUNIUS_DIR'):
    os.environ.pop(var, None)
!mkdir -p ~/.buildozer/android/platform/android-sdk
!SDKROOT=$HOME/.buildozer/android/platform/android-sdk; SDKMANAGER=$(find "$SDKROOT" -type f -name sdkmanager | head -n 1); if [ -z "$SDKMANAGER" ]; then rm -rf /tmp/android-cmdline-tools /tmp/android-cmdline-tools.zip; curl -fsSL https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip -o /tmp/android-cmdline-tools.zip; mkdir -p /tmp/android-cmdline-tools; unzip -q /tmp/android-cmdline-tools.zip -d /tmp/android-cmdline-tools; mkdir -p "$SDKROOT/cmdline-tools/latest"; cp -a /tmp/android-cmdline-tools/cmdline-tools/. "$SDKROOT/cmdline-tools/latest/"; SDKMANAGER="$SDKROOT/cmdline-tools/latest/bin/sdkmanager"; fi; yes | "$SDKMANAGER" --sdk_root="$SDKROOT" --licenses >/dev/null 2>&1; "$SDKMANAGER" --sdk_root="$SDKROOT" "platforms;android-33" "build-tools;33.0.2" "platform-tools"
!SDKROOT=$HOME/.buildozer/android/platform/android-sdk; test -f "$SDKROOT/platforms/android-33/android.jar" || { echo 'ERROR: Android API 33 was not installed'; exit 1; }; echo 'Verified Android API 33:'; ls -l "$SDKROOT/platforms/android-33/android.jar"
!yes | buildozer android clean
!rm -rf ~/.buildozer/android/platform/build-arm64-v8a_armeabi-v7a ~/.buildozer/android/packages
!export ANDROIDSDK=$HOME/.buildozer/android/platform/android-sdk; export ANDROID_HOME=$ANDROIDSDK; export ANDROID_SDK_ROOT=$ANDROIDSDK; export ANDROIDAPI=33; export ANDROIDMINAPI=21; set -o pipefail; buildozer -v android debug 2>&1 | tee /content/build.log

In [ ]:
# 6. Automatic download locator
import os
from google.colab import files

if os.path.exists('bin'):
    apk_files = [f for f in os.listdir('bin') if f.endswith('.apk')]
    if apk_files:
        target_apk = os.path.join('bin', apk_files[0])
        print(f'Downloading {target_apk}...克服')
        files.download(target_apk)
    else:
        print('No APK found inside the bin directory.')
else:
    print('Compilation was not successful or the bin directory is missing.')